# The model:
- Model parameters
- Update functions defining rules of behavior and transition between states.
- Update functions for the whole population of agents and households.


## Structs: Params and Simulation

1. Define a parameters struct and constructor called *Params*
- Number of households.
- Vulnerability threshold under which households become interested in migration.
- Total populatio.
- Individual income gain/loss ration threshold under which individuals become interested in migration.
- Cost of migration.
- Initial number of migrants.
- Random seed to ensure replicability.
- An empirical income distribution. Needs to be array. 
- Frequency weights based on histogram drawn from income distribution.

In [ ]:
mutable struct Params
    n_hh :: Int64 #Number of households
    vuln_th :: Float64 #Vulnerability threshold
    n :: Int64 #Total population
    p_contact :: Float64 #Proportion/Probability of community contact
    inc :: Float64 #Income gain loss ratio threshold
    cost :: Float64 #Migration cost
    nmig :: Int64 #Number of migrants
    seed :: Int64 #Random seed
    empInc :: Array  #Empirical income distribution.
    weights  #Frequency weights based on histogram drawn from income distribution. 
end
Params() = Params(1, 1, 1, 1, 1, 1, 0, 1, [], fw) #Default

2. Create a simulation *struct* with two properties.
- *pop*: Population of persons.
- *pop_hh*: Population of households.

In [ ]:
mutable struct Simulation
    pop :: Vector{Person}
    pop_hh :: Vector{Household}
end

## Update functions: two functions to acknowledge Theory of Planned Behavior. 
Not all individuals interested in migration will try to migrate (aspirations and capabilities).

1. **Households** transition from **state potential to intention**.
- If already interested in migration (status_hh == intention_hh), return.
- If vulnerability level higher than vulnerability threshold, household becomes interested in migrating. This draws from the New Economics of Labor Migration theory.
- Select a random member from household that will change status to intention.

In [ ]:
function update_potentialHH!(household, par :: Params)
    if household.status_hh == intention_hh
        return
    end
    if household.vuln > par.vuln_th
    household.status_hh = intention_hh
    end
    if household.status_hh == intention_hh
    rand(household.members).status = intention
    end
end

2. **Agents'** state transition from **potential to intention**: 
- Rules of behavior or agents rationale:
    - Agents are loss averse.
    - Only would want to migrate those were gains are relatively double larger than losses.
    - Intention is both affect by strong (mnet) and weak ties (cmnet), associated to households and communities, respectively.
    - Influence of mnet and cmnet varies randomly over time (*in review*).
    - mnet (representing hhs) has a greater influence than the community
    - Canonical option: multiply instead of divide (to avoid get the inverted version of the distribution)

In [4]:
function update_potential!(person, par :: Params)
    mnet = count(p -> p.status == migrant, person.family)
    cmnet =  count(p -> p.status == migrant, person.community)
    if (person.gain / person.loss) + mnet*rand(0.02:0.05) + cmnet*rand(0.01:0.02) > par.inc
        person.status = intention
    end
end

ArgumentError: ArgumentError: invalid type for argument par in method definition for update_potential! at c:\Users\ASUS\Dropbox\Documents\academico\PhD\MPIDR_RI_2023\tutorial_ABM\model.ipynb:1

3. **Agents'** state transition from intention **to actual migrants**:
- Compute size of household (mnet network). Object mnet counts number of migrants in agent's network.
- Compute total savings.
- Having household migrant networks (mnet) increases ability of migration, expressed in increased savings. This mechanisms needs to be improved. 

In [ ]:
function update_intention!(person, par :: Params)
    # Object mnet counts number of migrants in agent's network.
    mnet = count(p -> p.status == migrant,  person.family)
    #Agents save a proportion of their income in each time step.
    person.totsav = person.totsav + person.ainc*0.2
    #Migrant family have an additive effect on the capacity to migrate.
    #1 migrant contact increases in 1% the total savings, which represents the ability.
    if person.totsav + person.totsav*(mnet/100) > par.cost
        person.status = migrant
    end
end

4. Update all households and persons (agents).
- Update households and persons with the respective update functions applied to eachs unit of analysis (e.g. update_potentialHH!) (see above).

In [ ]:
function update_HH!(household, par :: Params)
    if isempty(household.members)
        return
    end
    if household.status_hh == potential_hh
       update_potentialHH!(household, par)
    end
end

In [ ]:
function update_AG!(person, par :: Params)
    if person.status == potential
       update_potential!(person, par)
   else
       update_intention!(person, par)
    end
end

5. Shuffle population of households and individuals within objects pop and pop_hh (part of Simulation struct).

In [ ]:

function update_households!(sim, par :: Params)
    orderHH = shuffle(sim.pop_hh)
    for hh in orderHH
        update_HH!(hh, par)
    end
end

function update_agents!(sim, par :: Params)
    order = shuffle(sim.pop)
    for p in order
        update_AG!(p, par)
    end
end